# Figures of the covariance-validation paper

Every figure, from the original NERSC files. Run section 0 once, then any figure cell; each cell makes one figure
from start to finish and saves it to `paper/figures/<name>.pdf` (used by `make paper`).

**The data.** `load_holi('LRG1', 'NGC')` returns one *sample* with plain attributes:

| attribute | what it is |
|---|---|
| `s.k` | $k$ bin centres ($\Delta k=0.005$, $0.02<k<0.3$: 56 bins) |
| `s.mocks` | every mock's $(P_0, P_2, P_4)$, one after the other: shape (number of mocks, 168) |
| `s.cov_mocks` | covariance of the mocks |
| `s.cov_gauss_local` | Gaussian covariance, local window $m^2$ (where we started) |
| `s.cov_gauss` | Gaussian covariance, $\xi$-kernel window |
| `s.cov_ssc`, `s.cov_disc` | super-sample and discreteness terms |
| `s.cov_final` | `cov_gauss + cov_ssc + cov_disc`: the recommended covariance (nothing fitted) |
| `s.cov_t0_tree`, `s.cov_t0_response` | connected trispectrum terms (rejected by the mocks) |
| `s.P0(v)`, `s.P2(v)`, `s.P4(v)` | cut a 168-vector (e.g. `np.diag(C)`) into its multipoles |

`load_abacus('complete' or 'altmtl', cap)`, `load_nz(tracer, cap)` and `load_randoms(tracer, cap)` work the same way.
The file reading is in `paperdata.py` (paths at its top), the statistics in `paperstats.py`; neither needs editing.

**Colours** mean the same thing in every figure: black = mocks, grey = Gaussian with the local window,
blue = Gaussian with the $\xi$-kernel window, orange = Gaussian + SSC + discreteness (recommended).

## 0. Setup (run once)

In [ ]:
import os, sys
HOME = os.path.expanduser('~')
sys.path.insert(0, HOME + '/thecov/paper')
sys.path.insert(0, HOME + '/thecov')
import numpy as np
import matplotlib.pyplot as plt
import paperdata
from paperdata import load_holi, load_abacus, load_nz, load_randoms, excluded_mocks
from paperstats import (correlation, chi2_of_each_mock, mean_chi2_per_dof, inverse_sqrt, whitened_mock_covariance,
                        whitened_trace, null_eigenvalues, wishart_sigma, variance_ratio_error, excess_directions,
                        project, fit_amplitudes, parameter_variance_ratio)
%matplotlib inline

FIGDIR = HOME + '/thecov/paper/figures'          # savefig() writes here
paperdata.CACHE_DIR = os.environ.get('SCRATCH', HOME) + '/paper_nersc_cache'   # None: always re-read the originals
paperdata.WORKERS = 32                             # processes reading the per-mock files
print('spectra from  ', paperdata.SPECTRA_BASE)
print('catalogues    ', paperdata.CATALOG_DIR)
print('covariances   ', paperdata.THECOV_CACHE['holi'], 'and', paperdata.OUT)

In [ ]:
# ---- look of every figure ----
MOCKS = '#0b0b0b'    # black : the mocks
LOCAL = '#898781'    # grey  : Gaussian, local window m^2
GAUSS = '#2a78d6'    # blue  : Gaussian, xi-kernel window
FINAL = '#eb6834'    # orange: Gaussian + SSC + discreteness (recommended)
SSC   = '#1baf7a'    # aqua  : the SSC term (figures about the terms)
DISC  = '#e87ba4'    # pink  : the discreteness term (always dashed too)
T0    = '#898781'    # grey  : connected trispectrum (rejected; always dotted)
LIGHT = '#e1e0d9'    # bands and grids

LOCAL_LABEL = 'Gaussian, local window'
GAUSS_LABEL = r'Gaussian, $\xi$-kernel window'
FINAL_LABEL = 'Gaussian + SSC + discreteness'
ELL_LINE = {0: '-', 2: '--', 4: ':'}                  # multipoles in one panel: line style
WIDTH = 6.3                                             # inches: the JCAP text width
K_LABEL = r'$k\ [h\,\mathrm{Mpc}^{-1}]$'

plt.rcParams.update({
    'font.family': 'serif', 'mathtext.fontset': 'cm', 'font.size': 9, 'axes.titlesize': 9, 'axes.labelsize': 9,
    'legend.fontsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.titlelocation': 'left',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#52514e',
    'axes.grid': True, 'grid.color': LIGHT, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'lines.linewidth': 1.6, 'lines.markersize': 4, 'legend.frameon': False, 'figure.dpi': 130,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.03, 'figure.constrained_layout.use': True})

def savefig(fig, name):
    os.makedirs(FIGDIR, exist_ok=True)
    fig.savefig(f'{FIGDIR}/{name}.pdf')
    print('wrote', f'{FIGDIR}/{name}.pdf')

In [ ]:
# ---- the four holi samples used by most figures (first run: reads every mock file, ~minutes) ----
lrg_ngc = load_holi('LRG1', 'NGC')
lrg_sgc = load_holi('LRG1', 'SGC')
qso_ngc = load_holi('QSO', 'NGC')
qso_sgc = load_holi('QSO', 'SGC')
for s in (lrg_ngc, lrg_sgc, qso_ngc, qso_sgc):
    print(f'{s.name}: {s.n_mocks} mocks, {s.n_k} k bins per multipole')
print('defective mocks left out (n(z) outliers): LRG1', excluded_mocks('LRG1'), ' QSO', excluded_mocks('QSO'))

## 1. The window (`window`)
How much the Gaussian variance changes when the window is pair-averaged with the $\xi$ kernel instead of taken
locally (left), and when each random carries its own weight instead of the local mean weight (right). LRG1 NGC.

In [ ]:
s = lrg_ngc
kernel_over_local = np.diag(s.cov_gauss) / np.diag(s.cov_gauss_local)
ownweight_over_local = np.diag(s.cov_gauss_ownweight) / np.diag(s.cov_gauss_local)

fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.4), sharey=True)
for ell in (0, 2, 4):
    left.plot(s.k, s.ell(kernel_over_local, ell), color=GAUSS, ls=ELL_LINE[ell], label=rf'$P_{ell}$')
    right.plot(s.k, s.ell(ownweight_over_local, ell), color=MOCKS, ls=ELL_LINE[ell], label=rf'$P_{ell}$')
left.set(title=r'$\xi$-kernel window / local window', xlabel=K_LABEL, ylabel='ratio of Gaussian variances')
right.set(title='own weight per random / local mean weight', xlabel=K_LABEL)
for ax in (left, right):
    ax.axhline(1, color=MOCKS, lw=0.7)
    ax.legend()
savefig(fig, 'window')

## 2. How big each non-Gaussian term is (`budget`)
Extra variance of $P_0$ from each term, in percent of the Gaussian variance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.5))
for ax, s in zip(axes, (lrg_ngc, qso_ngc)):
    gauss_P0 = s.P0(np.diag(s.cov_gauss))
    percent = lambda C: 100 * s.P0(np.diag(C)) / gauss_P0
    ax.plot(s.k, percent(s.cov_ssc), color=SSC, label='SSC')
    ax.plot(s.k, percent(s.cov_ssc_no_local_average), color=SSC, ls=':', label='SSC without the local average')
    ax.plot(s.k, percent(s.cov_disc), color=DISC, ls='--', label='discreteness')
    ax.plot(s.k, percent(s.cov_t0_response), color=T0, ls='-.', label=r'connected $T_0$ (rejected)')
    ax.axhline(0, color=MOCKS, lw=0.7)
    ax.set(title=s.name.replace('holi ', ''), xlabel=K_LABEL)
axes[0].set_ylabel(r'extra variance of $P_0$ [% of Gaussian]')
axes[0].legend(loc='upper left')
savefig(fig, 'budget')

## 3. What each term does (`anatomy`)
Left: eigenvalues of $C_{\rm G}^{-1/2}C_X C_{\rm G}^{-1/2}$ — a term with one large eigenvalue moves the data
vector in one direction only. Right: that leading direction for SSC, in units of the Gaussian error of each bin. LRG1 NGC.

In [ ]:
s = lrg_ngc
W = inverse_sqrt(s.cov_gauss)
fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.5))
for C, color, ls, label in ((s.cov_ssc, SSC, '-', 'SSC'), (s.cov_disc, DISC, '--', 'discreteness'),
                            (s.cov_t0_response, T0, ':', r'connected $T_0$')):
    eigenvalues = np.sort(np.abs(np.linalg.eigvalsh(W @ C @ W)))[::-1]
    left.plot(np.arange(1, 31), eigenvalues[:30], color=color, ls=ls, marker='o', ms=3, label=label)
left.set(yscale='log', ylim=(1e-4, 10), xlabel='rank', ylabel='eigenvalue (relative to Gaussian)', title='how many directions')
left.legend()

lam, vec = np.linalg.eigh(W @ s.cov_ssc @ W)
lead = np.linalg.inv(W) @ vec[:, -1] * np.sqrt(lam[-1]) / np.sqrt(np.diag(s.cov_gauss))   # in units of sigma_G
lead *= np.sign(lead[np.argmax(np.abs(lead))])
for ell in (0, 2, 4):
    right.plot(s.k, s.ell(lead, ell), color=SSC, ls=ELL_LINE[ell], label=rf'$P_{ell}$')
right.axhline(0, color=MOCKS, lw=0.7)
right.set(xlabel=K_LABEL, ylabel=r'shift / $\sigma_{\rm Gauss}$', title='the SSC direction')
right.legend()
savefig(fig, 'anatomy')

## 4. Diagonals (`diagonals_LRG1`, `diagonals_QSO`)
Variance of the mocks over the model variance, averaged over 4 neighbouring $k$ bins. Error bars: the scatter
expected from the finite number of mocks.

In [ ]:
GROUP = 4      # k bins averaged per point

def binned_variance_ratio(s, C, ell):
    # mean of (mock variance / model variance) over groups of GROUP bins of multipole ell: (k, ratio, error)
    ratio = s.ell(np.diag(s.cov_mocks) / np.diag(C), ell)
    first = {0: 0, 2: s.n_k, 4: 2 * s.n_k}[ell]
    k, r, err = [], [], []
    for g in range(s.n_k // GROUP):
        bins = np.arange(g * GROUP, (g + 1) * GROUP)
        k.append(s.k[bins].mean())
        r.append(ratio[bins].mean())
        err.append(variance_ratio_error(C, s.n_mocks, first + bins))
    return np.array(k), np.array(r), np.array(err)

for tracer, samples in (('LRG1', (lrg_ngc, lrg_sgc)), ('QSO', (qso_ngc, qso_sgc))):
    fig, axes = plt.subplots(2, 3, figsize=(WIDTH, 3.9), sharex=True, sharey=True)
    for row, s in enumerate(samples):
        for col, ell in enumerate((0, 2, 4)):
            ax = axes[row, col]
            for C, color, marker, shift, label in ((s.cov_gauss_local, LOCAL, 'o', -0.002, LOCAL_LABEL),
                                                   (s.cov_gauss, GAUSS, 's', 0.0, GAUSS_LABEL),
                                                   (s.cov_final, FINAL, 'D', 0.002, FINAL_LABEL)):
                k, r, err = binned_variance_ratio(s, C, ell)
                ax.errorbar(k + shift, r, err, fmt=marker, color=color, ms=3.5, lw=1, label=label)
            ax.axhline(1, color=MOCKS, lw=0.7)
            ax.set_title(rf'{s.name.split()[-1]}, $P_{ell}$')
            if col == 0:
                ax.set_ylabel('mock / model variance')
            if row == 1:
                ax.set_xlabel(K_LABEL)
    axes[0, 0].set_ylim(0.85, 1.3)
    fig.legend(*axes[0, 0].get_legend_handles_labels(), loc='lower center', ncol=3, bbox_to_anchor=(0.5, 1.0))
    savefig(fig, f'diagonals_{tracer}')

## 5. Veto holes (`holes`)
Left: fraction of the sky not vetoed, from the randoms (LRG1 NGC); the red box is shown enlarged in the middle.
Right: $R_{PP}$, the extra Gaussian variance that the variation of that fraction can add, against angular scale.

In [ ]:
import healpy as hp
ran = load_randoms('LRG1', 'NGC')
full_map = ran.fill_fraction_map(256)
zoom_map = ran.fill_fraction_map(1024)
lon, lat = hp.pix2ang(256, np.flatnonzero(full_map != hp.UNSEEN), lonlat=True)
ra_centre = np.degrees(np.angle(np.mean(np.exp(1j * np.radians(lon))))) % 360
zoom_centre = (ra_centre, float(np.median(lat)))          # change to zoom elsewhere
image_full = hp.projector.CartesianProj(rot=(ra_centre, 0), lonra=[-60, 60], latra=[-15, 85], xsize=900).projmap(
    full_map, lambda x, y, z: hp.vec2pix(256, x, y, z))
image_zoom = hp.projector.GnomonicProj(rot=zoom_centre, xsize=500, reso=4 * 60 / 500).projmap(
    zoom_map, lambda x, y, z: hp.vec2pix(1024, x, y, z))

fig, axes = plt.subplots(1, 3, figsize=(WIDTH, 2.4), gridspec_kw=dict(width_ratios=[1.5, 1, 1.1]))
im = axes[0].imshow(np.ma.masked_less(image_full, -1e30), origin='lower', cmap='Blues', vmin=0.5, vmax=1,
                    extent=(ra_centre + 60, ra_centre - 60, -15, 85), aspect='auto', interpolation='nearest')
axes[0].add_patch(plt.Rectangle((zoom_centre[0] - 2, zoom_centre[1] - 2), 4, 4, fill=False, color='#e34948', lw=1.2))
axes[0].set(xlabel='RA [deg]', ylabel='Dec [deg]')
fig.colorbar(im, ax=axes[0], location='top', fraction=0.06, pad=0.02, label='unvetoed fraction')
axes[1].imshow(np.ma.masked_less(image_zoom, -1e30), origin='lower', cmap='Blues', vmin=0, vmax=1.2,
               extent=(2, -2, -2, 2), interpolation='nearest')
axes[1].set(xlabel=r'$\Delta$RA [deg]', ylabel=r'$\Delta$Dec [deg]', title='zoom: 4° x 4°')
for ax in axes[:2]:
    ax.grid(False)
for tracer, cap, color, ls in (('LRG1', 'NGC', MOCKS, '-'), ('LRG1', 'SGC', MOCKS, '--'),
                               ('QSO', 'NGC', LOCAL, '-'), ('QSO', 'SGC', LOCAL, '--')):
    r = load_randoms(tracer, cap)
    axes[2].plot(r.scale_deg, r.R_PP, color=color, ls=ls, label=f'{tracer} {cap}')
axes[2].axhline(1, color=MOCKS, lw=0.7)
axes[2].set(xscale='log', xlabel='scale [deg]', ylabel=r'$R_{PP}$', title='extra variance from holes')
axes[2].legend(loc='upper left', fontsize=7)
savefig(fig, 'holes')

## 6. Correlation matrices and residuals (`corr`)
Top: correlation matrix of the mocks and of the recommended model. Bottom: mock minus model covariance in units
of its expected scatter, for the Gaussian and the recommended model (a correct model gives noise of unit size). LRG1 NGC.

In [ ]:
s = lrg_ngc
residual_gauss = (s.cov_mocks - s.cov_gauss) / wishart_sigma(s.cov_gauss, s.n_mocks)
residual_final = (s.cov_mocks - s.cov_final) / wishart_sigma(s.cov_final, s.n_mocks)
panels = [(correlation(s.cov_mocks), 'correlation: mocks', 'RdBu_r', 0.4),
          (correlation(s.cov_final), 'correlation: recommended model', 'RdBu_r', 0.4),
          (residual_gauss, r'(mocks $-$ Gaussian) / scatter', 'PuOr_r', 4),
          (residual_final, r'(mocks $-$ recommended) / scatter', 'PuOr_r', 4)]

fig, axes = plt.subplots(2, 2, figsize=(WIDTH, 5.6))
centres = [s.n_k / 2, 1.5 * s.n_k, 2.5 * s.n_k]
for ax, (M, title, cmap, vmax) in zip(axes.flat, panels):
    M = M.copy()
    np.fill_diagonal(M, np.nan)
    im = ax.imshow(M, cmap=cmap, vmin=-vmax, vmax=vmax, origin='lower', interpolation='nearest')
    for edge in (s.n_k, 2 * s.n_k):
        ax.axhline(edge - 0.5, color=MOCKS, lw=0.5)
        ax.axvline(edge - 0.5, color=MOCKS, lw=0.5)
    ax.set_xticks(centres, [r'$P_0$', r'$P_2$', r'$P_4$'])
    ax.set_yticks(centres, [r'$P_0$', r'$P_2$', r'$P_4$'])
    ax.tick_params(length=0)
    ax.grid(False)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
savefig(fig, 'corr')

## 7. $\chi^2$ of the mocks (`chi2`)
Left: $\chi^2$ of each LRG1 NGC mock against the theoretical $\chi^2$ distribution. Middle and right: average
$\chi^2$ per degree of freedom using only $k<k_{\max}$ (grey band: expected scatter).

In [ ]:
from scipy.stats import chi2 as chi2_distribution
KMAX_VALUES = np.arange(0.06, 0.301, 0.02)

fig, axes = plt.subplots(1, 3, figsize=(WIDTH, 2.4), gridspec_kw=dict(width_ratios=[1.2, 1, 1]))
s = lrg_ngc
n = s.mocks.shape[1]
bins = np.linspace(110, 330, 45)
for C, color, label in ((s.cov_gauss, GAUSS, 'Gaussian'), (s.cov_final, FINAL, 'recommended')):
    axes[0].hist(chi2_of_each_mock(s.mocks, C), bins=bins, histtype='step', color=color, lw=1.4, label=label)
x = np.linspace(110, 330, 300)
axes[0].plot(x, chi2_distribution.pdf(x, n) * s.n_mocks * (bins[1] - bins[0]), color=MOCKS, ls='--', lw=1,
             label=rf'$\chi^2_{{{n}}}$')
axes[0].set(xlabel=r'$\chi^2$', ylabel='mocks', title='LRG1 NGC', xlim=(100, 330))
axes[0].legend(loc='upper right', fontsize=7)

for ax, samples in ((axes[1], (lrg_ngc, lrg_sgc)), (axes[2], (qso_ngc, qso_sgc))):
    for s, ls in zip(samples, ('-', '--')):
        for C, color in ((s.cov_gauss, GAUSS), (s.cov_final, FINAL)):
            values = [mean_chi2_per_dof(s.mocks[:, s.below(kmax)], C[np.ix_(s.below(kmax), s.below(kmax))])
                      for kmax in KMAX_VALUES]
            ax.plot(KMAX_VALUES, values, color=color, ls=ls)
    n_dof = np.array([len(s.below(kmax)) for kmax in KMAX_VALUES])
    ax.fill_between(KMAX_VALUES, 1 - np.sqrt(2 / (n_dof * s.n_mocks)), 1 + np.sqrt(2 / (n_dof * s.n_mocks)),
                    color=LIGHT, lw=0)
    ax.axhline(1, color=MOCKS, lw=0.7)
    ax.set(xlabel=r'$k_{\max}$', ylim=(0.97, 1.12), title=s.name.split()[1])
axes[1].set_ylabel(r'$\langle\chi^2\rangle$ / dof')
axes[1].plot([], [], color=MOCKS, label='NGC')
axes[1].plot([], [], color=MOCKS, ls='--', label='SGC')
axes[1].legend(loc='upper left', fontsize=7)
savefig(fig, 'chi2')

## 8. Eigenvalues of the whitened mock covariance (`eigen`)
If a model is right, $C^{-1/2}SC^{-1/2}$ has the eigenvalues of pure noise. Plotted: each eigenvalue divided by
its noise expectation (grey band: 95% of noise realisations). Values above 1 are directions with too little model variance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.5), sharey=True)
for ax, s in zip(axes, (lrg_ngc, qso_ngc)):
    null = null_eigenvalues(s.n_mocks, s.mocks.shape[1])          # ~30 s
    null_median = np.median(null, axis=0)
    rank = np.arange(1, len(null_median) + 1)
    low, high = np.percentile(null, [2.5, 97.5], axis=0) / null_median
    ax.fill_between(rank, low, high, color=LIGHT, lw=0, label='noise only (95%)')
    for C, color, label in ((s.cov_gauss_local, LOCAL, LOCAL_LABEL), (s.cov_gauss, GAUSS, GAUSS_LABEL),
                            (s.cov_final, FINAL, FINAL_LABEL)):
        eig = np.sort(np.linalg.eigvalsh(whitened_mock_covariance(s.mocks, C)))[::-1]
        ax.plot(rank, eig / null_median, color=color, label=label)
    ax.axhline(1, color=MOCKS, lw=0.7)
    ax.set(xscale='log', yscale='log', xlabel='rank', title=s.name.replace('holi ', ''))
axes[0].set(ylabel='eigenvalue / noise expectation', ylim=(0.85, 3.2))
axes[0].set_yticks([0.9, 1, 1.5, 2, 3], ['0.9', '1', '1.5', '2', '3'])
axes[0].minorticks_off()
axes[1].legend(loc='upper right')
savefig(fig, 'eigen')

## 9. The mocks along the worst directions (`clouds`)
The mocks are split in two halves. The directions of largest excess variance are found in half A; half B is
then plotted along them, in units of the model's standard deviation — so a correct model gives a round cloud of
unit radius (circles: 1 and 2 $\sigma$). Right: the variance of half B along the first 15 such directions. LRG1 NGC.

In [ ]:
from matplotlib.patches import Circle, Ellipse
s = lrg_ngc
order = np.random.default_rng(7).permutation(s.n_mocks)
half_A, half_B = s.mocks[order[:s.n_mocks // 2]], s.mocks[order[s.n_mocks // 2:]]

def cloud(ax, C, color, title):
    _, directions, W = excess_directions(half_A, C)
    xy = project(half_B, W, directions[:, :2])
    ax.scatter(xy[:, 0], xy[:, 1], s=3, color=MOCKS, alpha=0.4, lw=0)
    for radius in (1, 2):
        ax.add_patch(Circle((0, 0), radius, fill=False, color=LOCAL, ls='--', lw=0.8))
    w, v = np.linalg.eigh(np.cov(xy.T))                    # 1-sigma ellipse of the mocks
    ax.add_patch(Ellipse((0, 0), 2 * np.sqrt(w[1]), 2 * np.sqrt(w[0]), angle=np.degrees(np.arctan2(v[1, 1], v[0, 1])),
                         fill=False, color=color, lw=1.8))
    var = xy.var(0, ddof=1)
    ax.text(0.03, 0.97, f'variances {var[0]:.2f}, {var[1]:.2f}', transform=ax.transAxes, va='top', fontsize=7.5)
    ax.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5), aspect='equal', title=title, xlabel='direction 1', ylabel='direction 2')

fig, axes = plt.subplots(1, 3, figsize=(WIDTH, 2.5), gridspec_kw=dict(width_ratios=[1, 1, 1.3]))
cloud(axes[0], s.cov_gauss, GAUSS, 'Gaussian')
cloud(axes[1], s.cov_final, FINAL, 'recommended')
N_DIRECTIONS = 15
for C, color, shift, label in ((s.cov_gauss, GAUSS, -0.15, 'Gaussian'), (s.cov_final, FINAL, 0.15, 'recommended')):
    _, directions, W = excess_directions(half_A, C)
    variance = project(half_B, W, directions[:, :N_DIRECTIONS]).var(0, ddof=1)
    axes[2].errorbar(np.arange(1, N_DIRECTIONS + 1) + shift, variance, variance * np.sqrt(2 / (len(half_B) - 1)),
                     fmt='o', color=color, ms=3.5, label=label)
axes[2].axhline(1, color=MOCKS, lw=0.7)
axes[2].set(xlabel='direction', ylabel='variance in half B', title='first 15 directions')
axes[2].legend()
savefig(fig, 'clouds')

## 10. Fitted amplitudes of the terms (`templates`)
Each term's amplitude fitted to the mocks ($C = C_{\rm G} + a_{\rm SSC}C_{\rm SSC} + a_{\rm disc}C_{\rm disc}\,[+\,a_{T_0}C_{T_0}]$);
1 = the prediction. Takes ~1 min.

In [ ]:
abacus_ngc, abacus_sgc = load_abacus('complete', 'NGC'), load_abacus('complete', 'SGC')
datasets = [lrg_ngc, lrg_sgc, qso_ngc, qso_sgc, abacus_ngc, abacus_sgc]
fit_two, fit_three = [], []                        # one ({term: amplitude}, {term: error}) per dataset
for s in datasets:
    fit_two.append(fit_amplitudes(s.mocks, s.cov_gauss, {'SSC': s.cov_ssc, 'disc': s.cov_disc}))
    fit_three.append(fit_amplitudes(s.mocks, s.cov_gauss, {'SSC': s.cov_ssc, 'disc': s.cov_disc, 'T0': s.cov_t0_response}))
    print(s.name, {t: f'{a:.2f} +- {fit_three[-1][1][t]:.2f}' for t, a in fit_three[-1][0].items()})

In [ ]:
XMIN, XMAX = -1, 3
names = [s.name.replace('holi ', '').replace('complete ', '') for s in datasets]
y = np.arange(len(datasets))[::-1]
fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.8), sharey=True)
for ax, fits, title in ((axes[0], fit_two, 'SSC + discreteness'), (axes[1], fit_three, r'SSC + discreteness + $T_0$')):
    for term, color, marker, dy in (('SSC', SSC, 'o', 0.15), ('disc', DISC, 's', 0.0), ('T0', T0, 'D', -0.15)):
        for yi, (amplitude, error) in zip(y, fits):
            if term not in amplitude:
                continue
            a, label = amplitude[term], {'SSC': 'SSC', 'disc': 'discreteness', 'T0': r'$T_0$'}[term] if yi == y[0] else None
            if XMIN < a < XMAX:
                ax.errorbar(a, yi + dy, xerr=error[term], fmt=marker, color=color, ms=4, label=label)
            else:                                   # off the axis: an arrow at the edge with the value
                edge = XMIN + 0.08 if a < XMIN else XMAX - 0.08
                ax.plot(edge, yi + dy, '<' if a < XMIN else '>', color=color, ms=5, label=label)
                ax.annotate(f'{a:.1f}', (edge, yi + dy), xytext=(6 if a < XMIN else -6, 0), textcoords='offset points',
                            ha='left' if a < XMIN else 'right', va='center', fontsize=7, color='#52514e')
    ax.axvline(1, color=MOCKS, lw=0.7)
    ax.axvline(0, color=LIGHT, lw=0.7)
    ax.set(xlim=(XMIN, XMAX), xlabel='fitted amplitude (1 = predicted)', title=title)
    ax.grid(axis='y', visible=False)
axes[0].set_yticks(y, names)
fig.legend(*axes[1].get_legend_handles_labels(), loc='lower center', ncol=3, bbox_to_anchor=(0.5, 1.0))
savefig(fig, 'templates')

## 11. Bin width (`binning`)
The same mocks measured with $\Delta k = 0.005, 0.01, 0.02$. Left: average mock/model variance over all directions.
Right: the largest whitened eigenvalue. A missing smooth non-Gaussian term grows with $\Delta k$; a missing
low-rank term keeps the same largest eigenvalue. LRG1.

In [ ]:
DK_VALUES = (0.005, 0.01, 0.02)
fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.4))
for cap, ls, face in (('NGC', '-', None), ('SGC', '--', 'white')):
    samples = [load_holi('LRG1', cap, dk) for dk in DK_VALUES]
    for model, color, label in (('cov_gauss_local', LOCAL, LOCAL_LABEL), ('cov_gauss', GAUSS, GAUSS_LABEL),
                                ('cov_final', FINAL, FINAL_LABEL)):
        trace = [whitened_trace(s.mocks, getattr(s, model)) for s in samples]
        largest = [np.linalg.eigvalsh(whitened_mock_covariance(s.mocks, getattr(s, model)))[-1] for s in samples]
        left.plot(DK_VALUES, trace, color=color, ls=ls, marker='o', mfc=face, label=label if cap == 'NGC' else None)
        right.plot(DK_VALUES, largest, color=color, ls=ls, marker='o', mfc=face)
left.axhline(1, color=MOCKS, lw=0.7)
left.set(xlabel=r'$\Delta k\ [h\,\mathrm{Mpc}^{-1}]$', ylabel=r'tr$(C^{-1}S)/n$', title='all directions (solid NGC, dashed SGC)')
right.set(xlabel=r'$\Delta k\ [h\,\mathrm{Mpc}^{-1}]$', ylabel='largest eigenvalue', title='worst direction')
for ax in (left, right):
    ax.set_xticks(DK_VALUES)
fig.legend(*left.get_legend_handles_labels(), loc='lower center', ncol=3, bbox_to_anchor=(0.5, 1.0))
savefig(fig, 'binning')

## 12. Fibre assignment (`fiber`)
25 AbacusSummit mocks with complete targets and with fibre assignment (altmtl), same initial conditions, each
compared with its own Gaussian covariance. Left: $\chi^2$ per dof of each mock. Right: variance ratio
(altmtl / complete) of the mocks divided by that of the models; 1 = fibre assignment changes the covariance
exactly as the model predicts. Error bars: bootstrap over mocks.

In [ ]:
K_RANGES = ((0.02, 0.1), (0.1, 0.2), (0.2, 0.3))
rng = np.random.default_rng(3)
fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.6), gridspec_kw=dict(width_ratios=[1, 1.6]))
for cap, face, shift in (('NGC', MOCKS, -0.08), ('SGC', 'white', 0.08)):
    complete, altmtl = load_abacus('complete', cap), load_abacus('altmtl', cap)
    n = complete.mocks.shape[1]
    chi2_complete = chi2_of_each_mock(complete.mocks, complete.cov_gauss_local) / n
    chi2_altmtl = chi2_of_each_mock(altmtl.mocks, altmtl.cov_gauss_local) / n
    left.scatter(chi2_complete, chi2_altmtl, s=16, color=MOCKS, facecolor=face, lw=0.8, label=cap)
    print(f'{cap}: <chi2/n> altmtl - complete = {np.mean(chi2_altmtl - chi2_complete):+.3f} '
          f'+- {np.std(chi2_altmtl - chi2_complete, ddof=1) / np.sqrt(len(chi2_altmtl)):.3f}')

    model_ratio = np.diag(altmtl.cov_gauss_local) / np.diag(complete.cov_gauss_local)
    def double_ratio(mocks_used):
        mock_ratio = altmtl.mocks[mocks_used].var(0, ddof=1) / complete.mocks[mocks_used].var(0, ddof=1)
        log_ratio = np.log(mock_ratio / model_ratio)
        return np.array([[np.exp(complete.ell(log_ratio, ell)[(complete.k >= lo) & (complete.k < hi)].mean())
                          for lo, hi in K_RANGES] for ell in (0, 2, 4)])
    value = double_ratio(np.arange(len(complete.mocks)))
    boot = np.array([double_ratio(rng.integers(0, len(complete.mocks), len(complete.mocks))) for _ in range(2000)])
    low, high = np.percentile(boot, [16, 84], axis=0)
    for j, (ell, marker) in enumerate(zip((0, 2, 4), ('o', 's', '^'))):
        x = np.arange(3) + (j - 1) * 0.25 + shift
        right.errorbar(x, value[j], [value[j] - low[j], high[j] - value[j]], fmt=marker, color=MOCKS, mfc=face, ms=4,
                       lw=1, label=rf'$P_{ell}$' if cap == 'NGC' else None)
left.plot((0.7, 1.75), (0.7, 1.75), color=LIGHT, lw=1)
left.set(xlim=(0.7, 1.75), ylim=(0.7, 1.75), aspect='equal', xlabel=r'$\chi^2$/dof, complete',
         ylabel=r'$\chi^2$/dof, altmtl', title='each mock')
left.legend()
right.axhline(1, color=MOCKS, lw=0.7)
right.set_xticks(range(3), [f'{lo} < k < {hi}' for lo, hi in K_RANGES])
right.set(ylim=(0.8, 1.2), ylabel='mocks / model', title='altmtl / complete variance (filled NGC, open SGC)')
right.legend(ncol=3)
savefig(fig, 'fiber')

## 13. Parameter errors (`params`)
Each mock fitted with each covariance for an amplitude $A$, an extra quadrupole amplitude $A_2$, a dilation
$\alpha$ and a monopole constant $N$. Plotted: scatter of the fits over the mocks / error predicted by the
covariance (variance ratio), against $k_{\max}$. Grey band: expected scatter. 1 = right error bars.

In [ ]:
KMAX_VALUES = np.arange(0.1, 0.301, 0.025)
PARAMETERS = (r'amplitude $A$', r'quadrupole $A_2$', r'dilation $\alpha$', r'constant $N$')
fig, axes = plt.subplots(2, 4, figsize=(WIDTH, 3.6), sharex=True, sharey=True)
for row, s in enumerate((lrg_ngc, qso_ngc)):
    for C, color, label in ((s.cov_gauss_local, LOCAL, LOCAL_LABEL), (s.cov_gauss, GAUSS, GAUSS_LABEL),
                            (s.cov_final, FINAL, FINAL_LABEL)):
        ratios = np.array([parameter_variance_ratio(s, C, kmax) for kmax in KMAX_VALUES])   # (n_kmax, 4)
        for col in range(4):
            axes[row, col].plot(KMAX_VALUES, ratios[:, col], color=color, marker='o', ms=3, label=label)
    for col in range(4):
        ax = axes[row, col]
        ax.axhspan(1 - np.sqrt(2 / (s.n_mocks - 1)), 1 + np.sqrt(2 / (s.n_mocks - 1)), color=LIGHT, lw=0)
        ax.axhline(1, color=MOCKS, lw=0.7)
        ax.set_title(PARAMETERS[col] if row == 0 else '')
        if row == 1:
            ax.set_xlabel(r'$k_{\max}$')
    axes[row, 0].set_ylabel(s.name.replace('holi ', '') + '\nvariance ratio')
axes[0, 0].set_ylim(0.8, 2.4)
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc='lower center', ncol=3, bbox_to_anchor=(0.5, 1.0))
savefig(fig, 'params')

## 14. Long radial modes in the mocks (`nz`)
Left: $n(z)$ of every LRG1 NGC mock relative to the median (black: the defective mocks left out). Right: scatter
of the galaxy counts in redshift slabs between mocks (shot noise removed), over the linear-theory prediction
$(b_1+f)\,\sigma_{\rm slab}$, against slab thickness. Below 1: the mocks lack long radial modes.

In [ ]:
from scipy.integrate import quad
OMEGA_M = 0.3153
def comoving_distance(z):        # [Mpc/h]
    return 2997.92458 * quad(lambda x: 1 / np.sqrt(OMEGA_M * (1 + x)**3 + 1 - OMEGA_M), 0, z)[0]

def slab_sigma(thickness, area, k, P):
    # rms of the mean density of a slab (thickness [Mpc/h], area [(Mpc/h)^2]) from radial modes
    q = np.linspace(1e-4, 2.0, 200000)
    return np.sqrt(np.trapezoid(np.interp(q, k, P) * np.sinc(q * thickness / 2 / np.pi)**2, q) / np.pi / area)

fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.5), gridspec_kw=dict(width_ratios=[1.1, 1]))
nz = load_nz('LRG1', 'NGC')
z_centres = 0.5 * (nz.z_edges[1:] + nz.z_edges[:-1])
relative = 100 * (nz.counts / np.median(nz.counts, axis=0) - 1)
defective = np.isin(nz.mock_ids, excluded_mocks('LRG1'))
for row in relative[~defective][:300]:
    left.plot(z_centres, row, color=LOCAL, lw=0.3, alpha=0.4)
for row in relative[defective]:
    left.plot(z_centres, row, color=MOCKS, lw=1)
left.set(xlabel='$z$', ylabel=r'$n(z)$ / median $-$ 1 [%]', title=f'LRG1 NGC ({defective.sum()} defective mocks)')

for tracer, color in (('LRG1', MOCKS), ('QSO', LOCAL)):
    s = load_holi(tracer, 'NGC')
    for cap, ls in (('NGC', '-'), ('SGC', '--')):
        nz = load_nz(tracer, cap)
        good = ~np.isin(nz.mock_ids, excluded_mocks(tracer))
        r0, r1 = comoving_distance(nz.z_edges[0]), comoving_distance(nz.z_edges[-1])
        area = nz.area_sr * (r1**3 - r0**3) / 3 / (r1 - r0)
        thickness, ratio = [], []
        for n_slabs in (20, 10, 5, 2, 1):
            counts = nz.counts[good].reshape(good.sum(), n_slabs, -1).sum(2)
            measured = np.sqrt(np.maximum((counts.std(0, ddof=1) / counts.mean(0))**2 - 1 / counts.mean(0), 0)).mean()
            predicted = (s.b1 + s.f) * slab_sigma((r1 - r0) / n_slabs, area, s.plin_k, s.plin)
            thickness.append((r1 - r0) / n_slabs)
            ratio.append(measured / predicted)
        right.plot(thickness, ratio, color=color, ls=ls, marker='o', ms=3.5, label=f'{tracer} {cap}')
right.axhline(1, color=MOCKS, lw=0.7)
right.set(xscale='log', xlabel=r'slab thickness [$h^{-1}$Mpc]', ylabel='mocks / linear theory', title='radial modes')
right.legend(ncol=2)
savefig(fig, 'nz')